In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load data
df = pd.read_csv('data/denial_labels_train.csv')

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=1000), 'denial_text'),
        ('code', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['denial_code'])
    ])

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['label'])

# Split
X_train, X_test, y_train, y_test = train_test_split(df.drop('label', axis=1), y, test_size=0.2, random_state=42)

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, multi_class='ovr'))
])

# Train
pipeline.fit(X_train, y_train)

# Predict & evaluate
y_pred = pipeline.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save
joblib.dump(pipeline, 'models/denial_classifier.pkl')
joblib.dump(le, 'models/label_encoder.pkl')


1.0
                   precision    recall  f1-score   support

  coding_bundling       1.00      1.00      1.00         4
      eligibility       1.00      1.00      1.00        17
medical_necessity       1.00      1.00      1.00        18
     missing_info       1.00      1.00      1.00        11
            other       1.00      1.00      1.00        13
    timely_filing       1.00      1.00      1.00        17
     underpayment       1.00      1.00      1.00         6

         accuracy                           1.00        86
        macro avg       1.00      1.00      1.00        86
     weighted avg       1.00      1.00      1.00        86



D:\Side_Projects\ERISA-AI-Portfolio-Challenge\env\lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


['label_encoder.pkl']